# 01 - Explorative Datenanalyse: Forex-Daten (Yahoo Finance)

**Ziel:** Forex-Kursdaten von Yahoo Finance laden, erkunden und erste Qualitätsprüfung durchführen.

**Währungspaare:** EUR/USD, EUR/CHF, GBP/USD, CHF/USD

**Datenquelle:** Yahoo Finance (via yfinance Library)

---

## 1. Setup und Imports

In [ ]:
# Bibliotheken importieren
import yfinance as yf          # Yahoo Finance API
import pandas as pd            # Datenverarbeitung
import numpy as np             # Numerische Berechnungen
import matplotlib.pyplot as plt # Visualisierung
import seaborn as sns           # Erweiterte Visualisierung
import os                       # Dateipfade

# Darstellung konfigurieren
plt.style.use('seaborn-v0_8')  # Schönerer Plot-Stil
plt.rcParams['figure.figsize'] = (12, 6)  # Standardgrösse für Plots
pd.set_option('display.max_columns', None)  # Alle Spalten anzeigen

print('Setup erfolgreich!')

## 2. Daten laden von Yahoo Finance

Wir laden die Forex-Kursdaten für unsere Währungspaare.
Yahoo Finance verwendet das Format `EURUSD=X` für Forex-Paare.

In [ ]:
# Konfiguration: Welche Währungspaare und welcher Zeitraum?
CURRENCY_PAIRS = {
    'EURUSD=X': 'EUR/USD',   # Euro zu US-Dollar
    'EURCHF=X': 'EUR/CHF',   # Euro zu Schweizer Franken
    'GBPUSD=X': 'GBP/USD',   # Britisches Pfund zu US-Dollar
}

START_DATE = '2024-01-01'
END_DATE = '2025-12-31'

print(f'Zeitraum: {START_DATE} bis {END_DATE}')
print(f'Währungspaare: {list(CURRENCY_PAIRS.values())}')

In [ ]:
# Daten laden und in einem Dictionary speichern
# Jedes Währungspaar wird separat geladen
forex_data = {}

for yahoo_symbol, pair_name in CURRENCY_PAIRS.items():
    print(f'Lade {pair_name} ({yahoo_symbol})...')
    
    # Daten herunterladen
    ticker = yf.Ticker(yahoo_symbol)
    df = ticker.history(start=START_DATE, end=END_DATE)
    
    # Im Dictionary speichern
    forex_data[pair_name] = df
    
    print(f'  -> {len(df)} Zeilen geladen')

print('\nAlle Daten geladen!')

## 3. Erste Datenübersicht

Schauen wir uns an, was wir bekommen haben.

In [ ]:
# Übersicht für jedes Währungspaar
for pair_name, df in forex_data.items():
    print(f'\n{"=" * 50}')
    print(f'{pair_name}')
    print(f'{"=" * 50}')
    print(f'Shape (Zeilen x Spalten): {df.shape}')
    print(f'Zeitraum: {df.index.min()} bis {df.index.max()}')
    print(f'Spalten: {list(df.columns)}')
    print(f'Datentypen:\n{df.dtypes}')
    print(f'\nErste 3 Zeilen:')
    display(df.head(3))

## 4. Datenqualitätsprüfung

Wir prüfen:
- Fehlende Werte (NaN)
- Duplikate
- Datentypen
- Statistische Kennzahlen

In [ ]:
# Fehlende Werte prüfen
print('FEHLENDE WERTE PRO WÄHRUNGSPAAR')
print('=' * 50)

for pair_name, df in forex_data.items():
    missing = df.isnull().sum()
    missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
    
    print(f'\n{pair_name}:')
    if missing.sum() == 0:
        print('  Keine fehlenden Werte!')
    else:
        for col in df.columns:
            if missing[col] > 0:
                print(f'  {col}: {missing[col]} fehlend ({missing_pct[col]}%)')

In [ ]:
# Duplikate prüfen (gleiche Datumswerte)
print('DUPLIKATE PRO WÄHRUNGSPAAR')
print('=' * 50)

for pair_name, df in forex_data.items():
    dupes = df.index.duplicated().sum()
    print(f'{pair_name}: {dupes} Duplikate')

In [ ]:
# Statistische Kennzahlen (Deskriptive Statistik)
for pair_name, df in forex_data.items():
    print(f'\n{"=" * 50}')
    print(f'Deskriptive Statistik: {pair_name}')
    print(f'{"=" * 50}')
    display(df.describe())

## 5. Visualisierung

### 5.1 Kursverlauf (Close-Preis)

In [ ]:
# Kursverlauf für alle Währungspaare plotten
fig, axes = plt.subplots(len(forex_data), 1, figsize=(14, 4 * len(forex_data)))

# Falls nur ein Paar, axes in Liste umwandeln
if len(forex_data) == 1:
    axes = [axes]

for ax, (pair_name, df) in zip(axes, forex_data.items()):
    ax.plot(df.index, df['Close'], label=pair_name, linewidth=1)
    ax.set_title(f'Kursverlauf {pair_name}', fontsize=14)
    ax.set_xlabel('Datum')
    ax.set_ylabel('Kurs')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 5.2 Verteilung der täglichen Renditen

In [ ]:
# Tägliche Renditen berechnen und Verteilung anzeigen
fig, axes = plt.subplots(1, len(forex_data), figsize=(6 * len(forex_data), 5))

if len(forex_data) == 1:
    axes = [axes]

for ax, (pair_name, df) in zip(axes, forex_data.items()):
    # Tägliche prozentuale Veränderung berechnen
    returns = df['Close'].pct_change().dropna()
    
    ax.hist(returns, bins=50, edgecolor='black', alpha=0.7)
    ax.set_title(f'Tägliche Renditen: {pair_name}', fontsize=12)
    ax.set_xlabel('Rendite (%)')
    ax.set_ylabel('Häufigkeit')
    ax.axvline(x=0, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

### 5.3 Fehlende Tage identifizieren (Wochenenden, Feiertage)

In [ ]:
# Lücken in den Daten identifizieren
for pair_name, df in forex_data.items():
    print(f'\n{pair_name}:')
    
    # Differenz zwischen aufeinanderfolgenden Tagen berechnen
    date_diffs = pd.Series(df.index).diff()
    
    # Lücken grösser als 1 Tag (ohne Wochenenden = 3 Tage)
    gaps = date_diffs[date_diffs > pd.Timedelta(days=3)]
    
    if len(gaps) > 0:
        print(f'  {len(gaps)} grössere Lücken gefunden (> 3 Tage):')
        for idx in gaps.index:
            print(f'    {df.index[idx-1].strftime("%Y-%m-%d")} -> {df.index[idx].strftime("%Y-%m-%d")} ({date_diffs[idx].days} Tage)')
    else:
        print('  Keine unerwarteten Lücken gefunden.')

## 6. Rohdaten speichern

Die Rohdaten werden unverändert als CSV gespeichert.

In [ ]:
# Rohdaten als CSV speichern
OUTPUT_DIR = '../data/raw/forex/yahoo'
os.makedirs(OUTPUT_DIR, exist_ok=True)

for pair_name, df in forex_data.items():
    # Dateinamen erstellen (z.B. EUR_USD_2024-01-01_to_2025-12-31.csv)
    safe_name = pair_name.replace('/', '_')
    filename = f'{safe_name}_{START_DATE}_to_{END_DATE}.csv'
    filepath = os.path.join(OUTPUT_DIR, filename)
    
    # Speichern
    df.to_csv(filepath)
    print(f'Gespeichert: {filepath} ({len(df)} Zeilen)')

print('\nAlle Rohdaten gespeichert!')

## 7. Zusammenfassung

### Erkenntnisse aus der EDA:
- **Datenumfang:** (hier Ergebnisse eintragen nach Ausführung)
- **Fehlende Werte:** (hier Ergebnisse eintragen)
- **Duplikate:** (hier Ergebnisse eintragen)
- **Auffälligkeiten:** (hier Ergebnisse eintragen)

### Nächste Schritte:
1. Gleiche Daten von EODHD laden
2. Datenqualität zwischen Yahoo und EODHD vergleichen
3. Daten bereinigen und harmonisieren